## 02-父子索引（Parent-Child Index）

### 离线构建流程
1. 对文档做层级切块：父块较大，子块较小；
2. 将子块向量化，存入向量数据库；
3. 将子块向量与原始父块文档通过 ID 关联，原始父块文档单独存储
4. 检索时，匹配子块向量，返回父块文档

#### 对话模型

In [19]:
import os
from langchain_openai import ChatOpenAI
model = ChatOpenAI(api_key=os.getenv("DASHSCOPE_API_KEY"), model="qwen3.7-max", base_url="https://ws-ots0tb33v08803wb.cn-beijing.maas.aliyuncs.com/compatible-mode/v1")

#### 嵌入模型

In [20]:
from langchain_community.embeddings import DashScopeEmbeddings
embeddings_model = DashScopeEmbeddings(model="text-embedding-v4", dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"))

# 通过本地ollama上运行的bge-m3做向量化, 使用需要关闭 代理/VPN
from langchain_ollama import OllamaEmbeddings
embeddings_model = OllamaEmbeddings(model="bge-m3")

#### 文档加载

In [21]:
from langchain_community.document_loaders import TextLoader
# 加载数据
loader = TextLoader("../../../data/deepseek百度百科.txt", encoding="utf-8")
docs = loader.load()
print(f"文章的长度：{len(docs[0].page_content)}")

文章的长度：20899


#### 文档切块

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 子块是父块内容的子集，1个父块分成4个子块
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1024) # 创建父文档分割器
child_splitter = RecursiveCharacterTextSplitter(chunk_size=256)   # 创建子文档分割器

#### 存储 + 检索构建
- 向量存储：Chroma, 存储子块向量
- 文档存储：InMemoryStore, 存储父块原始文档
- 检索器：ParentDocumentRetriever

In [23]:
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import ParentDocumentRetriever
# 创建向量数据库对象，存储子文档向量
vectorstore = Chroma(collection_name="split_parents", embedding_function = embeddings_model)
# 创建内存存储对象，存储原始父文档
store = InMemoryStore()
# 创建父子文档检索器，帮我们通过检索子块，返回父文档块
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore, # 向量存储对象
    docstore=store, # 文档存储对象
    child_splitter=child_splitter, # 子文档分割器，子文档存储到向量数据库
    parent_splitter=parent_splitter,# 主文档分割器，主文档存储到内存中
    search_kwargs={"k": 1}  # topK = 1, 只找出相似度最高的子文档块
)
# topK = 2,相似度最高的子文档块（A,B） A,B属于同一个父， 父文档块被查询两次，不会去重

#添加文档集
retriever.add_documents(docs)
print(f"主文块的数量：{len(list(store.yield_keys()))}")

主文块的数量：27


测试

In [24]:
# 测试 - 相似性搜索
'''这里我们通过向量数据库的similarity_search方法搜索出来的是与用户问题相关的子文档块的内容，
下面我们使用检索器的get_relevant_documents的方法来对这个问题进行检索，
它会返回该子文档块所属的主文档块的全部内容： '''
print("------------similarity_search------------------------")
sub_docs = vectorstore.similarity_search("deepseek的应用场景")
print('------------子块---------------')
print(sub_docs[0].page_content)
ids = [d.metadata[retriever.id_key] for d in sub_docs]
parent_docs = [d for d in retriever.docstore.mget(ids) if d is not None]
print('------------通过子块找到的父块---------------')
print(parent_docs[0].page_content)

print("------------检索器直接通过内部实现子找父-------------")
retrieved_docs = retriever.invoke("deepseek的应用场景")
print(retrieved_docs[0].page_content)
# # 测试 - 相似性搜索 - 完成
# exit()

------------similarity_search------------------------
------------子块---------------
DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据。由知名私募巨头幻方量化孕育而生，专注于开发先进的大语言模型（LLM）和相关技术。注册地址：浙江省杭州市拱墅区环城北路169号汇金国际大厦西1幢1201室。法定代表人为裴湉，经营范围包括技术服务、技术开发、软件开发等。
------------通过子块找到的父块---------------
DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据。由知名私募巨头幻方量化孕育而生，专注于开发先进的大语言模型（LLM）和相关技术。注册地址：浙江省杭州市拱墅区环城北路169号汇金国际大厦西1幢1201室。法定代表人为裴湉，经营范围包括技术服务、技术开发、软件开发等。
2024年1月5日至6月，相继发布DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2模型。 2024年9月5日，更新API支持文档，宣布合并DeepSeek Coder V2和DeepSeek V2 Chat，推出DeepSeek V2.5。12月13日，发布DeepSeek-VL2。12月26日，正式上线DeepSeek-V3首个版本并同步开源。2025年1月31日，英伟达宣布DeepSeek-R1模型登陆NVIDIANIM。同一时段内，亚马逊和微软也接入DeepSeek-R1模型。 2月5日，DeepSeek-R1、V3、Coder等系列模型，已陆续上线国家超算互联网平台。2月6日消息，澳大利亚政府以所谓“担心安全风险”为由，已禁止在所有政府设备中使用DeepSeek。2月8日，DeepSeek正式登陆苏州，并在苏州市公共算力服务平台上完成部署上线，为用户提供开箱即用的软硬件一体服务。 截至2月9日，DeepSeek App的累计下载量已超

拓展
- 方式一：找到子块就返回父文档块，问题：子块同属与一个父块，父文档备查询多次，需要去重
- 方式二：找到两个子块同属于一个父块，才返回父文档块

In [25]:
# 继承去重：继承 ParentDocumentRetriever
# 解决问题：topK = 2, 相似度最高的子文档块（A,B） A,B属于同一个父， 父文档块被查询两次
from langchain_classic.retrievers import ParentDocumentRetriever
class DedupParentDocumentRetriever(ParentDocumentRetriever):
    def _get_relevant_documents(self, query, *, run_manager):
        docs = super()._get_relevant_documents(query, run_manager=run_manager)
        seen, out = set(), []
        for d in docs:
            key = d.metadata.get("doc_id") or d.metadata.get("source")
            if key not in seen:
                seen.add(key)
                out.append(d)
        return out

#### 问题回答：增强 + 生成

In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap
from langchain_core.prompts import ChatPromptTemplate

# 创建prompt模板
template = """请根据下面给出的上下文来回答问题:
{context}
问题: {question}
"""

# 由模板生成prompt
prompt = ChatPromptTemplate.from_template(template)

#创建chain
chain = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
}) | prompt | model | StrOutputParser()

print("------------模型回复------------------------")

response = chain.invoke({"question": "deepseek的应用场景"})
print(response)

------------模型回复------------------------
根据提供的上下文，文中并未直接详细列举具体的细分业务场景（如智能客服、文章生成等），但通过其发布的模型类型、经营范围以及部署落地情况，可以总结出DeepSeek的以下应用方向和场景：

1. **通用语言处理与技术服务**：专注于开发先进的大语言模型（LLM），经营范围涵盖技术服务、技术开发等，可应用于各类自然语言处理场景。
2. **编程辅助与软件开发**：相继发布了DeepSeek-Coder、DeepSeek-Coder-V2等代码专属模型，可应用于代码生成、辅助编程及软件开发场景。
3. **数学计算与推理**：发布了DeepSeekMath模型，可应用于数学相关的计算、解题与逻辑推理场景。
4. **视觉与多模态理解**：发布了DeepSeek-VL和DeepSeek-VL2视觉语言模型，可应用于图像理解、图文多模态交互等场景。
5. **公共算力与软硬件一体化服务**：在苏州市公共算力服务平台完成部署，为用户提供“开箱即用的软硬件一体服务”；同时上线国家超算互联网平台，应用于算力调度与超算服务场景。
6. **企业级AI应用与云生态**：已有超200家企业宣布接入DeepSeek，同时英伟达（NVIDIA NIM）、亚马逊和微软等科技巨头也接入了其模型，广泛应用于各类企业的AI业务赋能和云服务生态中。
7. **个人移动端智能交互**：推出了DeepSeek App，供广大个人用户在移动端进行日常AI交互和使用（累计下载量超1.1亿次，周活跃用户近9700万）。
